<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week01_Turing_Test_Practical_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 1 Practical (PH):** Turing Test, the Chinese Room and your first AI animation

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO1, LO3

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — A rule-based chatbot (ELIZA-style) | Task A1 — Add **memory** and two new rules |
| Part B — Running a Turing test **scientifically** | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — The Chinese Room (Searle, 1980) | Task C1 — Extend the room, then break it |
| Part D — Human-like vs rational behaviour | Task D0 — The rational choice (class note §1.2) |
| Part E — AI technology and tools (LO1) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | build and extend a rule-based (ELIZA-style) chatbot | Apply | LO1 |
| 2 | run a Turing-test protocol and **test statistically** whether judges beat chance | Apply / Evaluate | LO3 |
| 3 | simulate Searle's **Chinese Room** and explain what it shows (and does not show) | Analyse | LO3 |
| 4 | contrast *human-like* behaviour with *rational* (optimal) behaviour | Analyse | LO3 |
| 5 | map AI branches to the Python tools you will use in this course | Understand | LO1 |

**Do Now (3 min, pairs):** *When might a rule-based chatbot fool you? When would it fail immediately?* Write one sentence in your Group Log.

## Part A — A rule-based chatbot (ELIZA-style)
A rule-based chatbot maps **patterns** in the user's text to **template replies**. ELIZA (Weizenbaum, 1966) showed that a handful of such rules can feel surprisingly human — which is exactly why the *Turing test* is a weak test of understanding.

### 🔍 Code walkthrough — how ELIZA decides what to say
Read the next cell as a short story with three characters:
1. **`rules`** is a list of *(pattern, handler)* pairs. A pattern is a regular expression, for example `\bi feel (.*)` which matches "I feel nervous" and *captures* the words after "feel".
2. **`eliza_reply`** goes through the rules **in order**. The first pattern that matches wins, so the order of the rules is part of the design.
3. If nothing matches, the bot picks a harmless fallback at random.

**Predict before you run:** what will the bot say to "I feel nervous about exams"? Which rule catches it, and what does `m.group(1)` contain?

In [ ]:
#@title 🗣️ Starter chatbot
# rules = (regular-expression pattern, handler function) pairs; ELIZA tries them in order
rules = [
    # (compiled regex, handler). The first rule that matches wins.
    (re.compile(r"\b(hello|hi|hey)\b", re.I),
     lambda m: random.choice(["Hello! How can I help today?", "Hi there — what's on your mind?"])),
    (re.compile(r"\bi feel (.*)", re.I),
     lambda m: f"Why do you feel {m.group(1).rstrip('.!?')}?"),
    (re.compile(r"(.*)\?\s*$", re.I),
     lambda m: "That's a good question. What do you think?"),
]
# Generic replies used when no pattern matches
FALLBACKS = ["Tell me more…", "How does that relate to your studies?", "What would make this better?"]

def eliza_reply(text: str) -> str:
    """Return a reply using the first matching rule, else a generic prompt."""
    # Remove leading/trailing spaces so patterns match cleanly
    text = text.strip()
    # Try each rule in order
    for pattern, handler in rules:
        # Search the sentence for the pattern; m is None when there is no match
        m = pattern.search(text)
        # If the pattern matched ...
        if m:
            # ... call its handler with the match to build the reply
            return handler(m)
    # No rule matched: pick a fallback at random
    return random.choice(FALLBACKS)

# Test the chatbot on four sample sentences
for s in ["hello", "I feel nervous about exams", "Is AI dangerous?", "The lecture was long"]:
    # Print each input and ELIZA's reply
    print(f"> {s}\n  {eliza_reply(s)}")

### Task A1 — Add **memory** and two new rules
1. If the user says *"my name is …"*, store the name and use it the next time they greet you.
2. Add **two** more rules for your scenario (e.g. thanks, bye, course-code questions).

The function must be called `eliza_reply_plus(text)`. Use the global `user_name`.

In [ ]:
# TODO(Student): finish eliza_reply_plus (memory + at least two new rules)
# Module-level variable that remembers the user's name between calls (the bot's memory)
user_name = None

# TODO: extend ELIZA so it remembers names and answers greetings
def eliza_reply_plus(text: str) -> str:
    # global lets the function change the module-level user_name
    global user_name
    # Remove leading/trailing spaces so patterns match cleanly
    text = text.strip()
    # 1) name memory      → store and greet
    # 2) greeting + name  → "Hi <Name>!"
    # 3) thanks / bye     → polite replies
    # Starter line: just reuse the basic ELIZA; replace it with your extended logic
    return eliza_reply(text)          # replace/extend this

In [ ]:
#@title ✅ Auto-check for Task A1
# Module-level variable that remembers the user's name between calls (the bot's memory)
user_name = None
# Test the name-memory and greeting behaviour
r1 = eliza_reply_plus("my name is ada"); r2 = eliza_reply_plus("hi")
# Test thanks and goodbye
r3 = eliza_reply_plus("thanks a lot");   r4 = eliza_reply_plus("ok bye")
check("remembers the name 'Ada'", lambda: "Ada" in r1 and "Ada" in r2, "store the name (title-cased) and reuse it")
check("answers thanks politely",  lambda: "welcome" in r3.lower())
check("says goodbye",             lambda: "goodbye" in r4.lower())

## Part B — Running a Turing test **scientifically**
Turing (1950) proposed the *imitation game*: a judge converses (by text) with a human and a machine and must say which is which. He predicted that by about the year 2000 an average interrogator would have **no more than a 70% chance** of making the right identification after five minutes of questioning.

**Protocol for your class (10–15 min)**
1. One student (the *hidden human*) and the chatbot answer the judge's questions through a shared screen, labelled **X** and **Y** (randomise which is which).
2. Each judge guesses which label is the machine. Record `correct = number of correct guesses`, `total = number of judges × rounds`.
3. Rotate roles and improve the bot with the feedback, then run again.

**Statistics.** If the judges cannot tell the difference, their accuracy follows Binomial(n, 0.5). The research question is *do judges identify the machine better than chance?*, so we test
$$H_0: p=\tfrac12 \quad\text{against}\quad H_1: p>\tfrac12$$
with an **exact one-sided binomial test**, $p_{\text{one-sided}}=\sum_{j=k}^{n}\binom{n}{j}(\tfrac12)^n$, and report a Wilson 95% confidence interval for the judges' accuracy. The **two-sided** test asks a different question (does performance *differ* from chance in either direction?) and gives twice the $p$-value when $k\ge n/2$. Choose the test that matches your question **before** you look at the data. (Class note, Example 1.2: 14 of 20 gives $\hat p=0.70$, Wilson CI $[0.481, 0.855]$, one-sided $p=0.058$, two-sided $p=0.115$.)

In [ ]:
# TODO(Student): complete turing_summary(correct, total) and enter YOUR class results below.
# SciPy supplies the exact binomial test and confidence interval
from scipy import stats

def turing_summary(correct, total, alpha=0.05):
    """Return accuracy, Wilson 95% CI, exact ONE-sided p-value (H1: p > 0.5), the two-sided p-value and a verdict."""
    # Placeholders: you will replace each None with a computed value
    acc = None; lo = hi = None; p = None; p2 = None; verdict = ""
    # Return everything as a dictionary so the check cells can read it
    return dict(accuracy=acc, ci=(lo, hi), p_value=p, p_two_sided=p2, verdict=verdict)

# Replace these with how many judges guessed correctly out of how many
class_correct, class_total = 13, 20     # ← replace with your class's real numbers
# Print the summary dictionary for the class data
print(turing_summary(class_correct, class_total))

In [ ]:
#@title ✅ Auto-check for Part B
# Test: 13 correct out of 20 judges
s = turing_summary(13, 20)
check("accuracy = 0.65",            lambda: abs(s["accuracy"] - 0.65) < 1e-9)
check("one-sided p-value ≈ 0.132",  lambda: abs(s["p_value"] - 0.1316) < 0.002, "use scipy.stats.binomtest(k, n, 0.5, alternative='greater')")
check("two-sided p-value ≈ 0.263 (twice the one-sided)", lambda: abs(s["p_two_sided"] - 0.2632) < 0.002)
# Test: 14 out of 20 (a borderline case)
s14 = turing_summary(14, 20)
check("note Example 1.2: 14/20 → one-sided p ≈ 0.058, not significant", lambda: abs(s14["p_value"] - 0.0577) < 0.002 and "no evidence" in s14["verdict"])
# Test: 17 out of 20 (clear evidence)
s17 = turing_summary(17, 20)
check("note Exercise 1.2: 17/20 → one-sided p ≈ 0.0013", lambda: abs(s17["p_value"] - 0.0013) < 0.0003)
check("Wilson CI contains 0.65",    lambda: s["ci"][0] < 0.65 < s["ci"][1])
# Test: 18 out of 20
s2 = turing_summary(18, 20)
check("18/20 judges → distinguishable", lambda: "distinguishable" in s2["verdict"] and s2["p_value"] < 0.05)

**Discuss (5 min).** (a) With only 20 judgements, how wide is the confidence interval? What sample size would you need to detect $p = 0.70$ vs $p = 0.50$ with 80% power? (b) Why is "failing to reject $H_0$" not the same as "the bot is intelligent"?

## Part C — The Chinese Room (Searle, 1980)
Searle imagines a person who understands **no Chinese** sitting in a room with a rulebook that maps incoming Chinese symbols to outgoing Chinese symbols. To observers outside, the room converses fluently. Searle's claim: manipulating symbols by rule (syntax) is not sufficient for understanding (semantics) — so passing a Turing-style test is not enough for *strong AI*.

Below, **you are the operator**. The rulebook is a Python dictionary; the code never "knows" what any symbol means.

In [ ]:
#@title 🏠 The Chinese Room (rulebook = dictionary)
# Searle's rulebook: maps Chinese input strings to Chinese output strings
RULEBOOK = {
    "你好吗？":       "我很好，谢谢。",          # "How are you?"          → "I'm fine, thanks."
    "你叫什么名字？": "我叫小明。",              # "What's your name?"     → "My name is Xiaoming."
    "你会说中文吗？": "当然，我会说中文。",      # "Do you speak Chinese?" → "Of course, I do."
}
# Default reply when the symbols are not in the rulebook ("Please say that again.")
DEFAULT = "请再说一遍。"                         # "Please say that again."

def room(symbols: str) -> str:
    """The operator looks the incoming string up in the rulebook. No meaning is involved."""
    # Look up the input and return the matching output; the person in the room understands nothing
    return RULEBOOK.get(symbols.strip(), DEFAULT)

# Ask three questions; the last is a paraphrase the rulebook has never seen
for q in ["你好吗？", "你叫什么名字？", "你的名字是什么？"]:     # the last one is a *paraphrase*
    # Print each question with the room's answer
    print(f"{q}  →  {room(q)}")

Notice the third question: it **means** the same as the second, but the room falls back to its default. A person who understands Chinese would not be fooled by a paraphrase; a lookup table is.

### Task C1 — Extend the room, then break it
1. Add **three** new question→answer pairs to `RULEBOOK` (use any language you know; the room must not "understand" them).
2. Write a one-line normaliser `normalise(s)` (lower-case, strip spaces/punctuation) and use it so that *superficial* variants still match.
3. Find a **paraphrase** that still defeats your room, and explain in two sentences what the room lacks.

Then discuss the classic replies: **systems reply** (the *whole system* understands), **robot reply** (ground symbols in sensors/actuators), **brain-simulator reply**, **other-minds reply**. Which one do you find strongest, and why?

In [ ]:
# TODO(Student): add 3 pairs to RULEBOOK and implement normalise() + room_plus()
# TODO: define a function that normalises text before lookup
def normalise(s: str) -> str:
    # Placeholder: return the text unchanged; add lower-casing and punctuation removal
    return s            # lower-case and strip spaces / punctuation here

# TODO: a smarter room that uses normalise()
def room_plus(symbols: str) -> str:
    # Placeholder: calls the plain room for now
    return room(symbols)

In [ ]:
#@title ✅ Auto-check for Task C1
check("rulebook has ≥ 6 entries",               lambda: len(RULEBOOK) >= 6)
check("normaliser removes case/punctuation",    lambda: normalise("  How ARE you?! ") == "how are you")
check("room_plus matches a superficial variant", lambda: room_plus("HOW ARE YOU??") != DEFAULT)
check("room_plus still falls back on a true paraphrase", lambda: room_plus("how do you do these days") == DEFAULT and room_plus("HOW ARE YOU??") != DEFAULT)

## Part D — Human-like vs rational behaviour
Russell & Norvig organise AI definitions along two axes: **thinking vs acting** and **humanly vs rationally**.

| | Human-like | Rational (optimal) |
|---|---|---|
| **Thinking** | cognitive modelling | laws of thought / logic |
| **Acting** | **Turing test** approach | **rational agent** approach |

A chatbot that imitates human typos is *acting humanly*; a navigation app that always minimises expected travel time is *acting rationally*. They can disagree: a human-like agent may choose a familiar, sub-optimal route. In the next cell you compare two agents chasing a target: a **random walker** (no goal) and a **greedy** agent (rational given its goal). You measure *steps-to-goal* over 100 seeds and report mean ± 95% CI.

### Task D0 — The rational choice (class note §1.2)
A **rational** agent picks $a^*=\arg\max_a\sum_{s'}P(s'\mid s,a)\,U(s')$. Four assumptions sit behind this: a **goal** (the utility $U$), **information** about the state, reasonable **probabilities**, and a **performance measure** applied from outside. Example 1.1 in the note: action $a_1$ pays $10$ with probability $0.6$; action $a_2$ pays $5$ with probability $0.9$. Implement `best_action` and use it on the route problem of Exercise 1.1 (route A: 30 min w.p. 0.7, 60 min w.p. 0.3; route B: 40 min for sure; $U=-\text{time}$).

In [ ]:
# TODO(Student): expected utility and the rational action
def expected_utility(outcomes):
    """outcomes: list of (probability, utility). Return the expected utility."""
    # Placeholder return; your code replaces this line
    return None

def best_action(actions):
    """actions: dict name -> list of (probability, utility). Return (best_name, {name: EU})."""
    # Placeholder return (no best action, empty table); replace with your result
    return None, {}

In [ ]:
#@title ✅ Auto-check for Task D0
# Two bets: a1 risky but higher EV, a2 safer
bet = {"a1": [(0.6, 10), (0.4, 0)], "a2": [(0.9, 5), (0.1, 0)]}
# Ask the agent for the rational choice
name, eu = best_action(bet)
check("Example 1.1: a1 is rational (EU 6 vs 4.5)", lambda: name == "a1" and abs(eu["a1"] - 6) < 1e-9 and abs(eu["a2"] - 4.5) < 1e-9)
# Two routes: A is uncertain, B is certain
routes = {"A": [(0.7, -30), (0.3, -60)], "B": [(1.0, -40)]}
# Choose the better route by expected utility
name, eu = best_action(routes)
check("Exercise 1.1: route A (EU -39 vs -40)", lambda: name == "A" and abs(eu["A"] + 39) < 1e-9)
# Reminder: rational is not the same as lucky
print("A rational choice can still be unlucky: expected utility is an average, not a guarantee.")

In [ ]:
# TODO(Student): implement step_toward(pos, goal, speed) → new position (move at most `speed` straight toward goal)
# TODO: move the agent one step towards the goal
def step_toward(pos, goal, speed=0.5):
    # Placeholder: the agent stays where it is; replace with a step towards the goal
    return pos

# Run one episode and count steps until the agent reaches the goal
def steps_to_goal(policy, seed, max_steps=500, tol=0.5):
    # Seeded random generator so results are reproducible
    rng = np.random.default_rng(seed)
    # Random start position and random goal inside a 10x10 world
    pos = rng.uniform(0, 10, 2); goal = rng.uniform(0, 10, 2)
    # Allow at most max_steps moves
    for t in range(1, max_steps + 1):
        # Ask the policy for the next position
        pos = policy(pos, goal, rng)
        # Reached the goal if the distance is below the tolerance
        if np.linalg.norm(pos - goal) < tol:
            # Report how many steps it took
            return t
    # Never reached the goal: report the maximum
    return max_steps

# Random policy: wander with Gaussian noise, staying inside the world
random_policy = lambda pos, goal, rng: np.clip(pos + rng.normal(0, 0.5, 2), 0, 10)
# Greedy policy: always step towards the goal
greedy_policy = lambda pos, goal, rng: step_toward(pos, goal, 0.5)

In [ ]:
#@title 📊 Compare the two agents (same 100 start/goal pairs → paired test)
# Run the random policy on 100 seeded worlds
r = [steps_to_goal(random_policy, s) for s in range(100)]
# Run the greedy policy on the same 100 worlds (paired comparison)
g = [steps_to_goal(greedy_policy, s) for s in range(100)]
# Print mean, standard deviation and 95% CI for both
report("random walker  (steps)", r); report("greedy agent   (steps)", g)
# Paired Wilcoxon test: is the greedy agent faster in the same worlds?
print("Paired Wilcoxon signed-rank p =", f"{paired_wilcoxon(r, g):.2e}")
check("greedy reaches the goal in < 40 steps on average", lambda: np.mean(g) < 40)
check("greedy is significantly faster (p < 0.001)",       lambda: paired_wilcoxon(r, g) < 1e-3)

### 🎞️ Mini animation (CCMAS lab: *simple AI animation exercises*)
The cell below animates the greedy agent. In Colab the animation appears inline.
**Extension:** add a second, randomly moving *predator* and make the agent flee when it comes within distance 2.

In [ ]:
#@title 🎞️ Animate the greedy agent
# Matplotlib base package (needed for animation)
import matplotlib
# Plotting interface
import matplotlib.pyplot as plt
# Tool to build animations frame by frame
from matplotlib.animation import FuncAnimation
# Lets us show the animation inside the notebook
from IPython.display import HTML

# Build an animation of the greedy agent
def make_animation(seed=3, frames=40):
    # Seeded random generator so results are reproducible
    rng = np.random.default_rng(seed)
    # Random start position and random goal inside a 10x10 world
    pos = rng.uniform(0, 10, 2); goal = rng.uniform(0, 10, 2)
    # Record the agent's starting position
    path = [pos.copy()]
    # Simulate a fixed number of moves
    for _ in range(frames):
        # Move one step and save the new position
        pos = step_toward(pos, goal, 0.5); path.append(pos.copy())
    # Convert the list to an array for fast slicing
    path = np.array(path)
    # Create the plotting window and fix its axes
    fig, ax = plt.subplots(figsize=(4, 4)); ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    # Draw the goal as a red star
    ax.plot(*goal, "r*", ms=15, label="goal")
    # Two artists: a blue trail and a dot for the agent
    trail, = ax.plot([], [], "b-", lw=1); dot, = ax.plot([], [], "bo")
    # Add legend and title
    ax.legend(loc="upper right"); ax.set_title("Greedy agent")
    # Function called by the animation to draw frame i
    def update(i):
        # Show the trail up to frame i and the dot at frame i
        trail.set_data(path[:i+1, 0], path[:i+1, 1]); dot.set_data([path[i, 0]], [path[i, 1]])
        # Return the objects that changed
        return trail, dot
    # Wire everything together: one frame every 80 ms
    anim = FuncAnimation(fig, update, frames=len(path), interval=80, blit=True)
    # Close the static figure so only the animation displays
    plt.close(fig)
    # Return the animation object
    return anim

# Animation needs a notebook front end, so try it and carry on if it fails
try:
    # Build the animation
    anim = make_animation()
    # Convert it to HTML and display it
    display(HTML(anim.to_jshtml()))
# If it fails, we log a warning and continue
except Exception as e:                       # animation needs a notebook front-end; don't crash the lab
    # Write the reason to the log instead of crashing the lab
    log.warning("animation skipped: %s", e)

## Part E — AI technology and tools (LO1)
Match each branch of AI to a Python tool you will use this semester.

In [ ]:
#@title 🧰 Toolbox inventory
# Which AI tools belong to which branch of the field (data for the table)
TOOLBOX = {
    "Search & planning":              ["heapq", "collections.deque", "networkx"],
    "Knowledge representation/logic": ["our own unifier + resolution (Week 5)", "networkx (Week 6)"],
    "Machine learning":               ["scikit-learn", "numpy", "scipy"],
    "Natural language processing":    ["re", "scikit-learn (TF-IDF)", "nltk"],
    "Computer vision":                ["OpenCV (cv2)", "scikit-image", "numpy"],
    "Multi-agent / simulation":       ["numpy", "matplotlib.animation"],
}
# importlib lets us import a library from its name as a string
import importlib
# Go through each branch and its tools
for branch, tools in TOOLBOX.items():
    # Print a neat two-column row
    print(f"{branch:<34} {', '.join(tools)}")
# Blank line for readability
print()
# Check that each core library is installed
for lib in ["numpy", "scipy", "sklearn", "matplotlib", "networkx"]:
    # Animation needs a notebook front end, so try it and carry on if it fails
    try:
        # Import it and print its version
        mod = importlib.import_module(lib); print(f"{lib:<12} {getattr(mod, '__version__', '?')}")
    # If the library is missing ...
    except ImportError:
        # ... say so instead of crashing
        print(f"{lib:<12} not installed")
check("numpy and scikit-learn are available", lambda: importlib.util.find_spec("numpy") and importlib.util.find_spec("sklearn"))

## 📝 Assignment (5 marks, individual, formative)
1. *(2)* Extend `eliza_reply_plus` with **two new rules of your own** and paste five transcripts that show them working.
2. *(2)* Run a Turing-style test with **at least 20 judges** (classmates or friends). Report accuracy, the 95% Wilson interval, the one-sided exact binomial *p*-value and **three sentences** on what the result does and does not show.
3. *(1)* In ≤150 words, explain what your `normalise()` fix changes about the Chinese Room: is the person in the room any closer to *understanding*?

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection (5–7 sentences)
- When was your chatbot convincing and when did it break? What does that tell you about the Turing test as a measure of intelligence?
- Which reply to Searle (systems, robot, brain-simulator, other minds) do you find most convincing, and why?
- Give one example where *human-like* and *rational* behaviour would disagree in a Nigerian setting (e.g. a mobile-money assistant, a campus guide).

```text
Write your reflection here…
```

## Extensions
- Add a topic tracker (if the user mentions *exam*, ask a follow-up) and keep the last 3 messages as context.
- Replace the dictionary FAQ in your bot with TF-IDF retrieval (preview of Week 7).
- Read Turing (1950), *Computing Machinery and Intelligence*, §1–6 and list two objections he anticipated.

In [ ]:
checks_summary()  # how many auto-checks have you passed?